# pose-image-tool — 참조 이미지의 **자세만** 가져와 새 이미지 만들기

사진 속 인물의 **포즈(관절 구조)** 만 뽑아내서, 인물·의상·배경·화풍은 내가 쓴 글(프롬프트)대로
완전히 새로 그려내는 노트북입니다.

| 항목 | 내용 |
| --- | --- |
| 실행 환경 | Google Colab **무료 T4 GPU** |
| 사용 모델 | Stable Diffusion 1.5 + ControlNet v1.1 (OpenPose) |
| 첫 실행 소요 시간 | 약 6~9분 (설치 2분 + 모델 다운로드 4~6분) |
| 두 번째부터 | 이미지 한 장당 **10~20초** |
| 비용 | 무료 |

---

## 이 노트북이 하는 일

```
   ① 참조 사진             ② 스켈레톤(뼈대)            ③ 최종 결과
 ┌───────────────┐      ┌───────────────┐      ┌───────────────┐
 │  팔 든 사람   │ ───▶ │ 검은 배경 위   │ ───▶ │ 갑옷 입은 기사 │
 │   (내 사진)   │      │  색깔 점과 선  │      │ (프롬프트대로) │
 └───────────────┘      └───────────────┘      └───────────────┘
                          OpenPose가 추출        Stable Diffusion이 생성
                                                  ＋ ControlNet이 자세 강제
```

**자세는 참조 사진에서, 나머지 전부(누가·무엇을 입고·어디서·어떤 화풍)는 프롬프트에서** 옵니다.
참조 사진의 얼굴·피부색·옷은 **결과에 전혀 전달되지 않습니다.** 관절 좌표만 넘어갑니다.

---

## 전체 단계 지도

| STEP | 하는 일 | 걸리는 시간 | 다시 실행해야 할 때 |
| --- | --- | --- | --- |
| **0** | GPU 켜기 | 30초 | 런타임 새로 연결할 때마다 |
| **1** | 라이브러리 설치 | 2분 | 런타임 새로 연결할 때마다 |
| **2** | 라이브러리 불러오기 | 10초 | 런타임 새로 연결할 때마다 |
| **3** | 참조 사진 올리기 | 30초 | 다른 사진을 쓸 때 |
| **4** | 사진 열기 · 크기 계산 | 즉시 | 사진을 바꿀 때 |
| **5** | 자세(스켈레톤) 추출 | 첫 1분 / 이후 3초 | 사진을 바꿀 때 |
| **6** | 생성 모델 불러오기 | 첫 5분 / 이후 10초 | 모델을 바꿀 때 |
| **7** | 프롬프트·설정 정하기 | 즉시 | **실험할 때마다** |
| **8** | 이미지 생성 | 10~20초 | **실험할 때마다** |
| **9** | 저장 · 비교 · 다운로드 | 5초 | 결과를 남길 때 |
| **10** | 여러 장 한 번에 뽑기 | 장당 15초 | 선택 사항 |

> 한 번 다 돌리고 나면, 보통 **STEP 7 → STEP 8 → STEP 9 만 반복**하게 됩니다.
> 앞 단계(모델 로딩)는 메모리에 그대로 남아 있어서 다시 할 필요가 없습니다.

---

## 노트북을 처음 써 보는 분께

* **셀(cell)** — 이 문서를 이루는 네모 칸 하나하나입니다. 회색 배경에 코드가 적힌 칸이 **코드 셀**,
  지금 읽고 있는 설명 칸이 **텍스트 셀**입니다.
* **실행 방법** — 코드 셀을 클릭한 뒤 **`Shift` + `Enter`**, 또는 셀 왼쪽의 **▶ 버튼**을 누릅니다.
* **끝났는지 확인** — 실행 중이면 왼쪽에 **`[*]`** 또는 회전하는 원, 끝나면 **`[1]`, `[2]`** 처럼
  숫자가 붙습니다. **숫자가 붙을 때까지 기다린 뒤** 다음 셀로 가세요.
* **순서 지키기** — 이 노트북은 **위에서부터 순서대로** 실행하도록 만들어져 있습니다.
  건너뛰면 아래 셀에서 `NameError` 가 납니다.
* **주석 해제** — 코드 앞에 `#` 이 붙은 줄은 "실행되지 않는 메모"입니다.
  쓰고 싶으면 `#` 을 지우면 됩니다. 여러 줄을 한 번에 바꾸려면 드래그 후 **`Ctrl` + `/`**.
* **꼬였을 때** — 상단 메뉴 **런타임 → 세션 다시 시작 및 모두 실행** 으로 처음부터 다시 돌립니다.

문제가 생기면 맨 아래 **문제 해결** 표와 **용어 사전**을 보세요.

---
# STEP 0 — GPU 런타임 켜기

**이 노트북에서 가장 중요한 단계입니다.** GPU 없이 돌리면 이미지 한 장에 10분 이상 걸립니다.

## 0-1. 설정 방법

1. 상단 메뉴 **런타임(Runtime)** 클릭
2. **런타임 유형 변경(Change runtime type)** 클릭
3. **하드웨어 가속기(Hardware accelerator)** 를 **`T4 GPU`** 로 선택
4. **저장(Save)** 클릭 — 런타임이 다시 시작될 수 있습니다. 정상입니다.

> **T4가 무엇인가요?**
> Colab 무료 계정에 배정되는 NVIDIA 그래픽카드입니다. 메모리 약 15GB로,
> 이 노트북이 쓰는 모델(약 3.5GB)을 돌리기에 넉넉합니다.
> GPU는 같은 계산을 수천 개씩 동시에 처리해서, 그림 생성 같은 작업에서 CPU보다 수십 배 빠릅니다.

## 0-2. 제대로 붙었는지 확인

아래 셀을 실행하세요. `nvidia-smi` 는 그래픽카드 상태를 보여 주는 명령어입니다.
(코드 앞의 `!` 는 "파이썬 코드가 아니라 터미널 명령어"라는 표시입니다.)

In [ ]:
# [STEP 0-2] GPU가 붙었는지 확인
!nvidia-smi

### 결과 읽는 법

| 화면에 보이는 것 | 뜻 | 할 일 |
| --- | --- | --- |
| `Tesla T4` 와 `15360MiB` | 정상 | STEP 1로 |
| `L4` 또는 `A100` | 더 좋은 GPU | STEP 1로 |
| `nvidia-smi: command not found` | **GPU 꺼짐** | 0-1로 돌아가 설정 |

표의 `Memory-Usage` 칸이 `0MiB / 15360MiB` 라면 아직 아무것도 안 올린 상태 — 정상입니다.

---
# STEP 1 — 필요한 라이브러리 설치

Colab에는 기본 도구만 깔려 있어서, 이미지 생성에 필요한 라이브러리를 직접 설치해야 합니다.

> **주의**: 설치한 내용은 **런타임이 끊기면 전부 사라집니다.**
> 나중에 다시 접속하면 이 STEP부터 다시 실행해야 합니다.

## 무엇을 설치하나요?

| 라이브러리 | 하는 일 | 쉽게 말하면 |
| --- | --- | --- |
| `diffusers` | Stable Diffusion 계열 모델을 실행하는 엔진 | **그림 그리는 기계** |
| `transformers` | 프롬프트(글)를 모델이 이해하는 숫자 벡터로 변환 | **통역사** |
| `accelerate` | 모델을 GPU에 효율적으로 배치 | **짐 싣는 기사** |
| `controlnet_aux` | 사진에서 관절·윤곽선 등을 추출하는 도구 모음 | **자세 관찰자** |
| `huggingface_hub` | 인터넷에서 모델 파일을 내려받는 통로 | **배달부** |
| `safetensors` | 모델 파일을 안전·고속으로 읽는 파일 형식 | **압축 해제기** |

## 버전을 고정한 이유

이 라이브러리들은 업데이트가 잦아서 **최신 버전끼리 서로 안 맞는 일이 흔합니다.**
특히 `huggingface_hub` 1.0 이상은 `diffusers 0.31` 과 호환되지 않아 오류가 납니다.
아래 조합은 T4에서 동작이 확인된 조합이니 **버전 숫자를 바꾸지 마세요.**

## 1-1. 라이브러리 설치 (1~2분)

`-q` 는 "quiet"의 약자로, 설치 로그를 최소한만 출력하라는 뜻입니다.

In [ ]:
# [STEP 1-1] 라이브러리 설치 (1~2분 소요)
!pip install -q "diffusers==0.31.0" "transformers==4.46.3" "accelerate==1.1.1" "controlnet_aux==0.0.9" "huggingface_hub<1.0" "safetensors>=0.4.2"
print("pip 설치 명령 종료 — 아래 1-2 셀로 넘어가세요.")

### 결과 읽는 법 — **빨간 글씨가 뜨는 것이 정상입니다**

#### 1. `ERROR: pip's dependency resolver ...` (gradio 관련)

아래와 거의 똑같은 문구가 빨간 글씨로 나옵니다. **무시하고 진행하세요.**

```
ERROR: pip's dependency resolver does not currently take into account all the
packages that are installed. This behaviour is the source of the following
dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0,
but you have huggingface-hub 0.36.2 which is incompatible.
```

**무슨 일이 일어난 건가요?**

| | |
| --- | --- |
| **`gradio`** | Colab에 **기본으로 깔려 있는** 웹 UI 제작용 라이브러리. 최신 `huggingface_hub`(1.16 이상)를 요구합니다. |
| **우리가 한 일** | `huggingface_hub` 를 **일부러 1.0 미만(0.36.2)으로 낮췄습니다.** |
| **왜 낮췄나** | `huggingface_hub` 1.0부터 함수 일부가 삭제되어 **`diffusers 0.31`이 동작하지 않습니다.** 둘을 동시에 만족시킬 수 없어 한쪽을 골라야 합니다. |
| **왜 괜찮은가** | **이 노트북은 `gradio`를 한 번도 쓰지 않습니다.** `import gradio` 가 없으므로 아무 영향이 없습니다. |

`ERROR` 라고 적혀 있지만 실제로는 **경고**입니다. pip은 설치를 정상적으로 마쳤습니다.
정말 설치가 됐는지는 바로 아래 **1-3 셀**에서 숫자로 확인합니다.

> 같은 런타임에서 나중에 `gradio` 를 쓰는 다른 노트북 코드를 돌리면 그때는 실제로 깨집니다.
> 그럴 일이 있다면 런타임을 새로 시작한 뒤 따로 실행하세요.

#### 2. `RESTART SESSION` 버튼

화면 아래에 **`RESTART SESSION`(세션 다시 시작)** 버튼이 뜨면 **누르지 말고 그냥 진행**하세요.
이 노트북은 재시작 없이 동작합니다. 이미 눌렀다면 STEP 1-1부터 다시 실행하면 됩니다.

## 1-2. 한글 폰트 설치 *(그래프 글자 깨짐 방지)*

STEP 9에서 결과를 비교할 때 그래프에 **한글 제목**을 붙입니다.
그런데 Colab에는 한글 폰트가 없어서, 설치하지 않으면 제목이 **□□□** 네모로 깨져 보입니다.
나눔고딕 폰트를 미리 깔아 둡니다. (약 10초, 실패해도 노트북은 정상 동작합니다.)

In [ ]:
# [STEP 1-2] 한글 폰트 설치 (그래프 제목이 네모로 깨지는 것 방지, 약 10초)
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1
print("한글 폰트 설치 시도 완료 (실패해도 노트북은 영어 제목으로 정상 동작합니다)")

## 1-3. 설치 결과 확인

정말로 원하는 버전이 깔렸는지 눈으로 확인합니다.
여기서 `!! 설치되지 않음` 이 하나라도 보이면 1-1을 다시 실행하세요.

In [ ]:
# [STEP 1-3] 설치된 버전 확인
from importlib.metadata import PackageNotFoundError, version

EXPECTED = {
    "torch": "Colab 기본 제공 (2.x면 정상)",
    "diffusers": "0.31.0 이어야 함",
    "transformers": "4.46.3 이어야 함",
    "accelerate": "1.1.1 이어야 함",
    "controlnet_aux": "0.0.9 이어야 함",
    "huggingface_hub": "1.0 미만이어야 함 (0.3x면 정상)",
}

print(f"{'라이브러리':<18}{'설치된 버전':<16}기대값")
print("-" * 72)
missing = []
for pkg, note in EXPECTED.items():
    try:
        print(f"{pkg:<18}{version(pkg):<16}{note}")
    except PackageNotFoundError:
        print(f"{pkg:<18}{'!! 설치되지 않음':<16}{note}")
        missing.append(pkg)

print()
if missing:
    print(f"설치되지 않은 패키지가 있습니다: {missing}")
    print("STEP 1-1 셀을 다시 실행하세요.")
else:
    # 가장 흔한 실패 원인이라 따로 확인합니다.
    hub_major = int(version("huggingface_hub").split(".")[0])
    if hub_major >= 1:
        print("문제: huggingface_hub이 1.0 이상입니다. diffusers 0.31과 호환되지 않습니다.")
        print("      STEP 1-1 셀을 다시 실행하세요.")
    else:
        print("모든 라이브러리 준비 완료 — STEP 2로 넘어가세요.")
        print("(1-1에서 gradio 관련 빨간 ERROR가 떴더라도, 위 목록이 맞으면 정상입니다.)")

---
# STEP 2 — 라이브러리 불러오기 & 계산 장치 정하기

설치한 도구를 **메모리에 올리고**, 계산을 어디서(`device`) 어떤 정밀도로(`dtype`) 할지 정합니다.

## `device` — 계산을 어디서 할까

* `"cuda"` = NVIDIA GPU. **CUDA**는 NVIDIA가 만든 GPU 계산 규격의 이름입니다.
* `"cpu"` = 일반 프로세서. 여기로 떨어지면 매우 느립니다.

## `dtype` — 숫자를 몇 비트로 저장할까

모델은 수십억 개의 숫자(가중치)로 이루어져 있습니다. 이 숫자 하나를 몇 비트로 저장하느냐에 따라
메모리 사용량과 속도가 결정됩니다.

| 자료형 | 비트 | 메모리 | 속도 | 품질 |
| --- | --- | --- | --- | --- |
| `float32` | 32비트 | 기준 | 기준 | 기준 |
| `float16` | 16비트 | **절반** | **약 2배** | 차이 거의 없음 |

T4는 `float16` 전용 연산 회로(**Tensor Core**)를 갖고 있어서, **여기서는 `float16`이 정답**입니다.
GPU가 없을 때만 어쩔 수 없이 `float32`로 떨어집니다. (CPU는 `float16`을 제대로 못 다룹니다.)

> 참고: `bfloat16` 이라는 자료형도 있지만 T4(Turing 세대)는 지원하지 않습니다. 쓰면 오히려 느려집니다.

## 2-1. 라이브러리 불러오기

`import` 는 "이 도구를 지금부터 쓰겠다"는 선언입니다.
`as np` 처럼 별명을 붙이면 이후 `numpy` 대신 `np` 로 짧게 부를 수 있습니다.

In [ ]:
# [STEP 2-1] 라이브러리 불러오기

import glob      # 폴더 자동 탐색
import os        # 파일 경로 다루기
import shutil    # 폴더 압축하기 (STEP 10에서 사용)
import time      # 걸린 시간 재기

import numpy as np          # 숫자 배열 다루기 (이미지를 숫자로 볼 때 사용)
import torch                # 딥러닝 계산 엔진
from PIL import Image, ImageOps   # 이미지 열기/자르기/회전

# Colab 환경인지 확인합니다. (업로드/다운로드 기능을 쓸 수 있는지 판단용)
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("라이브러리 불러오기 완료")
print(f"Colab 환경 여부: {IN_COLAB}")

## 2-2. GPU 연결 확인 및 자료형 결정

여기서 정한 `DEVICE` 와 `DTYPE` 는 STEP 6, 8에서 계속 쓰입니다.

In [ ]:
# [STEP 2-2] 계산 장치와 자료형 결정

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"PyTorch 버전 : {torch.__version__}")
print(f"계산 장치    : {DEVICE}")
print(f"자료형       : {DTYPE}")

if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU 이름     : {props.name}")
    print(f"GPU 메모리   : {props.total_memory / 1024**3:.1f} GB")
    print("\nGPU 준비 완료 — STEP 3으로 넘어가세요.")
else:
    print("\n" + "!" * 64)
    print("경고: GPU가 잡히지 않았습니다.")
    print("이대로 진행하면 이미지 한 장에 10분 이상 걸립니다.")
    print("STEP 0으로 돌아가 [런타임 > 런타임 유형 변경 > T4 GPU]를 설정한 뒤,")
    print("STEP 1부터 다시 실행하세요.")
    print("!" * 64)

---
# STEP 3 — 참조할 포즈 사진 준비

자세를 가져올 **원본 사진**이 필요합니다.

## 먼저 알아 둘 것 — 코드는 내 PC가 아니라 구글 서버에서 돌아갑니다

```
[내 PC]                              [구글 서버 / Colab VM]
 VS Code 또는 브라우저 화면            실제 파이썬 실행 (Linux)
 C:\  G:\  바탕화면                    /content  /usr  ...
      │                                      │
      └──── 코드를 보내고 결과를 받음 ───────┘
                 (파일은 자동으로 안 넘어감)
```

**내 PC에 있는 사진은 구글 서버에서 보이지 않습니다.** 그래서 사진을 서버로 넘기는 방법이 따로 필요합니다.

| 순서 | 방법 | 되는 환경 | 셀 |
| --- | --- | --- | --- |
| **1순위** | **구글 드라이브 연결** | VS Code · Colab 웹 모두 | **3-1** |
| 2순위 | URL로 내려받기 | 어디서나 확실히 동작 | 3-2 |
| 3순위 | 내 컴퓨터에서 업로드 | **Colab 웹 브라우저 전용** | 3-3 |
| — | 로컬 경로 직접 지정 | 내 PC에서 직접 실행할 때만 | 3-4 |

> **3-3 업로드 방식 주의**
> `files.upload()` 는 Colab 웹페이지의 자바스크립트로 파일 선택 창을 띄웁니다.
> **VS Code에서는 그 자바스크립트가 없어 셀이 영원히 멈춥니다.**
> 몇 분이 지나도 `[*]` 가 그대로면 정지 버튼으로 중단하고 3-1이나 3-2를 쓰세요.

## 좋은 참조 사진 고르는 법

**잘 되는 사진**

* 사람이 **한 명**만 있고, 몸 전체(또는 상반신)가 **잘리지 않고** 보이는 사진
* 팔다리가 몸통에 **가려지지 않은** 자세 — 뒤로 숨긴 손은 관절을 찾지 못합니다
* 인물이 화면에서 **충분히 큰** 사진 (화면의 절반 이상 차지하면 이상적)

**잘 안 되는 사진**

* 여러 명이 겹쳐 있는 사진 → 뼈대가 뒤엉킵니다
* 인물이 아주 작게 찍힌 풍경 사진 → 관절을 못 찾습니다
* 극단적인 각도(바로 위에서 내려다본 사진 등) → 관절 추정이 흔들립니다

**상관없는 것** — 사진의 화질, 색감, 배경의 복잡함, 인물의 얼굴·성별·옷.
어차피 **관절 좌표만** 뽑아 쓰고 나머지는 전부 버립니다.

## 3-0. 지금 코드가 어디서 실행되는지 확인

어떤 방법을 써야 할지 이 셀이 알려 줍니다. 3초 안에 끝납니다.

In [ ]:
# [STEP 3-0] 실행 환경 확인

import platform

is_remote = os.path.exists("/content")

print(f"운영체제       : {platform.system()}")
print(f"현재 작업 경로 : {os.getcwd()}")
print(f"/content 존재  : {is_remote}")
print("-" * 64)

if is_remote:
    print("원격 Colab 서버(Linux)에서 실행 중입니다.")
    print("내 PC의 파일은 여기서 보이지 않습니다.")
    print()
    print("-> 다음: 3-1 (구글 드라이브 연결) 셀을 실행하세요.")
    print("   막히면 3-2 (URL)를 쓰세요.")
else:
    print("내 PC에서 직접 실행 중입니다.")
    print("프로젝트 폴더의 파일을 바로 쓸 수 있습니다.")
    print()
    print("-> 다음: 3-4 (로컬 경로 직접 지정) 셀을 쓰세요.")

## 3-1. 구글 드라이브 연결해서 사진 가져오기 *(1순위)*

이 프로젝트 폴더 `내 드라이브/Aiffel_Work/pose-image-tool` 은 **이미 구글 드라이브에 있습니다.**
Colab 서버가 같은 드라이브를 연결하면 `samples/` 폴더의 사진을 그대로 쓸 수 있습니다.

### 실행하면 일어나는 일

1. **`Permit this notebook to access your Google Drive files?`** 같은 인증 요청이 뜹니다 → 허용
2. 구글 계정 선택 → 권한 승인
3. `Mounted at /content/drive` 가 출력되면 연결 성공
4. `samples/` 폴더의 사진 목록이 자동으로 뜨고, **첫 번째 사진이 자동 선택**됩니다

**고칠 것이 딱 하나 있습니다.** 다른 사진을 쓰고 싶으면 맨 윗줄 `PICK = 0` 의
숫자만 바꾸고 셀을 다시 실행하세요. (`0` = 첫 번째, `1` = 두 번째)

> **1~2분이 지나도 인증창이 안 뜨고 `[*]` 그대로라면** VS Code에서 인증이 막힌 것입니다.
> 정지 버튼으로 중단하고 **3-2 (URL)** 로 넘어가세요.

In [ ]:
# [STEP 3-1] 구글 드라이브 연결해서 사진 가져오기  <- 1순위

PICK = 0   # samples 폴더의 몇 번째 사진을 쓸지 (0 = 첫 번째, 1 = 두 번째 ...)

POSE_IMAGE = None

from google.colab import drive

drive.mount("/content/drive")   # 인증 요청이 뜨면 허용해 주세요.

# 프로젝트 폴더를 내 드라이브에서 자동으로 찾습니다.
# 폴더를 다른 위치로 옮겨도(3단계 깊이까지) 경로를 고칠 필요가 없습니다.
PROJECT_NAME = "pose-image-tool"
DRIVE_ROOT = "/content/drive/MyDrive"


def find_project(root=DRIVE_ROOT, name=PROJECT_NAME, max_depth=3):
    """내 드라이브 아래에서 name 폴더를 찾아 경로를 돌려준다. 못 찾으면 None."""
    for depth in range(max_depth + 1):
        pattern = os.path.join(root, *(["*"] * depth), name)
        for hit in sorted(glob.glob(pattern)):
            if os.path.isdir(os.path.join(hit, "samples")):
                return hit
    return None


PROJECT_DIR = find_project()

if PROJECT_DIR:
    print(f"프로젝트 폴더를 찾았습니다: {PROJECT_DIR}")
else:
    # 못 찾으면 이 줄의 경로를 직접 고쳐 주세요.
    PROJECT_DIR = os.path.join(DRIVE_ROOT, "Aiffel_Work", PROJECT_NAME)
    print(f"자동 탐색 실패. 기본 경로를 씁니다: {PROJECT_DIR}")

SAMPLES_DIR = os.path.join(PROJECT_DIR, "samples")

print()
if os.path.isdir(SAMPLES_DIR):
    # 대소문자 구분 없이 이미지 확장자만 골라냅니다. (.PNG 도 인식)
    IMG_EXTS = (".png", ".jpg", ".jpeg", ".webp")
    found = sorted(
        os.path.join(SAMPLES_DIR, name)
        for name in os.listdir(SAMPLES_DIR)
        if name.lower().endswith(IMG_EXTS)
    )
    if found:
        print(f"samples 폴더에서 사진 {len(found)}개를 찾았습니다.")
        for i, f in enumerate(found):
            mark = "   <- 선택됨" if i == PICK % len(found) else ""
            print(f"  [{i}] {os.path.basename(f)}{mark}")

        POSE_IMAGE = found[PICK % len(found)]
        print(f"\nPOSE_IMAGE = '{POSE_IMAGE}'")
        print("\n다른 사진을 쓰려면 맨 위 PICK 숫자를 바꾸고 이 셀을 다시 실행하세요.")
        print("새 사진을 쓰려면 내 PC의 samples 폴더에 넣고 동기화되기를 기다린 뒤 다시 실행하세요.")
    else:
        print(f"폴더는 찾았지만 사진이 없습니다: {SAMPLES_DIR}")
        print("samples 폴더에 png/jpg 파일을 넣고 다시 실행하세요.")
else:
    print(f"경로를 찾지 못했습니다: {SAMPLES_DIR}")
    root = "/content/drive/MyDrive"
    if os.path.isdir(root):
        print(f"\n참고 — 내 드라이브 최상위 폴더 목록:")
        for name in sorted(os.listdir(root))[:30]:
            print(f"  {name}")
        print("\n위 목록을 보고 PROJECT_DIR 경로를 고친 뒤 다시 실행하세요.")

## 3-2. URL에서 내려받기 *(2순위 — 3-1이 막혔을 때)*

자바스크립트도 구글 인증도 필요 없어서 **어떤 환경에서든 확실히 동작합니다.**

### 이미지 주소 얻는 법

브라우저에서 원하는 사진 위에 **오른쪽 클릭 → 이미지 주소 복사**
(영어 메뉴면 `Copy image address`). 주소가 `.jpg`, `.png` 등으로 끝나면 좋습니다.

### 고칠 것

아래 셀의 `IMAGE_URL = ""` 에서 **따옴표 사이**에 복사한 주소를 붙여넣고 실행하세요.
비워 두고 실행하면 아무 일도 일어나지 않으니, 3-1이 성공했다면 그냥 넘어가도 됩니다.

In [ ]:
# [STEP 3-2] URL에서 사진 내려받기  <- 3-1이 안 될 때만

IMAGE_URL = ""   # <<< 여기 따옴표 사이에 이미지 주소를 붙여넣으세요

if not IMAGE_URL:
    print("IMAGE_URL이 비어 있어 아무것도 하지 않았습니다.")
    print("3-1(드라이브 연결)이 성공했다면 이 셀은 건너뛰어도 됩니다.")
else:
    import requests

    POSE_IMAGE = "reference.jpg"
    resp = requests.get(IMAGE_URL, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
    resp.raise_for_status()   # 주소가 잘못되면 여기서 오류가 납니다.
    with open(POSE_IMAGE, "wb") as f:
        f.write(resp.content)

    print(f"다운로드 완료 ({len(resp.content) / 1024:.0f} KB)")
    print(f"POSE_IMAGE = '{POSE_IMAGE}'")

## 3-3. 내 컴퓨터에서 업로드 — **Colab 웹 브라우저 전용** *(3순위)*

**VS Code에서는 쓰지 마세요.** 파일 선택 창이 뜨지 않고 셀이 영원히 멈춥니다.

Colab 웹(`colab.research.google.com`)에서 이 노트북을 열었다면 아래 `#` 을 지우고 쓰면 됩니다.
(드래그한 뒤 `Ctrl` + `/`)

In [ ]:
# [STEP 3-3] 내 컴퓨터에서 업로드 — Colab 웹에서만 동작 (VS Code에서는 멈춥니다)

# from google.colab import files
#
# uploaded = files.upload()
# POSE_IMAGE = list(uploaded.keys())[0]
# print(f"업로드 완료 -> POSE_IMAGE = '{POSE_IMAGE}'")

## 3-4. 로컬 경로 직접 지정 — 내 PC에서 직접 실행할 때만

3-0에서 **`Windows`** 또는 **`Darwin`(macOS)** 이 나왔다면 이 방법을 씁니다.
이 경우 노트북과 사진이 같은 컴퓨터에 있으므로 경로만 적으면 됩니다.

In [ ]:
# [STEP 3-4] 로컬 경로 직접 지정 (내 PC에서 직접 실행할 때만)

# POSE_IMAGE = "samples/pose_01.png"
# print(f"POSE_IMAGE = '{POSE_IMAGE}'")

---
# STEP 4 — 사진 열기 · 만들 이미지 크기 정하기

## 4-1. 사진 열기

파일을 실제로 읽어서 화면에 보여 줍니다. **업로드가 제대로 됐는지 눈으로 확인하는 단계**입니다.

코드에서 하는 세 가지 처리:

| 코드 | 하는 일 | 왜 필요한가 |
| --- | --- | --- |
| `Image.open()` | 파일을 이미지 객체로 읽기 | — |
| `ImageOps.exif_transpose()` | **휴대폰 사진의 회전 정보 반영** | 이게 없으면 세로 사진이 옆으로 누워서 처리됩니다 |
| `.convert("RGB")` | 투명 배경·흑백을 일반 컬러로 통일 | PNG 투명 영역이나 흑백 사진에서 오류 나는 것 방지 |

In [ ]:
# [STEP 4-1] 참조 사진 열기

# STEP 3을 건너뛰었을 때 친절한 안내를 띄웁니다.
if "POSE_IMAGE" not in globals() or POSE_IMAGE is None:
    raise RuntimeError("POSE_IMAGE가 없습니다. STEP 3-1(드라이브) 또는 3-2(URL)를 먼저 실행하세요.")
if not os.path.exists(POSE_IMAGE):
    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {POSE_IMAGE}\nSTEP 3-1을 다시 실행하세요.")

try:
    pose_src = Image.open(POSE_IMAGE)
except Exception as e:
    raise RuntimeError(
        f"이미지로 읽을 수 없는 파일입니다: {POSE_IMAGE}\n"
        f"jpg / png / webp 형식의 사진을 올렸는지 확인하세요. (원본 오류: {e})"
    )

pose_src = ImageOps.exif_transpose(pose_src)   # 휴대폰 사진 회전 보정
pose_src = pose_src.convert("RGB")             # 컬러 형식 통일

print(f"파일       : {POSE_IMAGE}")
print(f"원본 크기  : {pose_src.size[0]} x {pose_src.size[1]} 픽셀")
print(f"가로세로비 : {pose_src.size[0] / pose_src.size[1]:.2f}  "
      f"({'가로로 김' if pose_src.size[0] > pose_src.size[1] else '세로로 김' if pose_src.size[0] < pose_src.size[1] else '정사각형'})")

# 셀의 마지막 줄에 변수 이름만 쓰면 Colab이 그 값을 화면에 보여 줍니다. (이미지면 그림으로)
pose_src

### 결과 읽는 법

* 올린 사진이 그대로 보이면 정상입니다.
* 사진이 **옆으로 누워** 보이면 → `exif_transpose` 로도 못 잡은 경우입니다.
  원본을 직접 회전시켜 저장한 뒤 STEP 3부터 다시 하세요.
* `FileNotFoundError` → STEP 3-1을 다시 실행하세요.

## 4-2. *(선택)* 인물 부분만 잘라내기

**스켈레톤이 잘 안 잡히는 이유의 1순위는 "인물이 너무 작게 찍혀서"** 입니다.
사진에서 인물 주변만 잘라내면 관절 인식률이 크게 올라갑니다.

일단 4-1 결과가 괜찮아 보이면 **이 셀은 건너뛰고** STEP 4-3으로 가세요.
STEP 5에서 뼈대가 엉망으로 나왔을 때 돌아와서 쓰면 됩니다.

**좌표 읽는 법** — 이미지의 **왼쪽 위가 (0, 0)** 이고, 오른쪽으로 갈수록 `x` 증가,
아래로 갈수록 `y` 증가합니다. `CROP_BOX = (왼쪽, 위, 오른쪽, 아래)` 순서로 픽셀 값을 적습니다.

예: 800×1200 사진에서 가운데 세로 기둥만 남기려면 → `(200, 0, 600, 1200)`

In [ ]:
# [STEP 4-2] (선택) 인물 부분만 잘라내기 — 쓸 때만 아래 # 을 지우세요
#
# 예시: samples/pose_01.png (1280x720 애니 캡처)에서 인물만 남기려면
#       CROP_BOX = (150, 0, 1050, 720)   -> 900x720 으로 좁혀집니다.

# CROP_BOX = (150, 0, 1050, 720)    # (왼쪽, 위, 오른쪽, 아래) 픽셀 좌표
#
# before = pose_src.size
# pose_src = pose_src.crop(CROP_BOX)
# print(f"자르기 전: {before[0]} x {before[1]}")
# print(f"자른 후  : {pose_src.size[0]} x {pose_src.size[1]}")
# print()
# print("중요: 크기가 바뀌었으므로 아래 STEP 4-3 셀을 반드시 다시 실행하세요.")
# pose_src

## 4-3. 만들 이미지 크기 자동 계산

**왜 크기를 따로 계산하나요?**

참조 사진이 세로로 길쭉한데 결과를 정사각형(512×512)으로 만들면
**다리가 잘리거나 몸이 옆으로 퍼져 보입니다.** 그래서 원본의 가로세로 비율을 그대로 유지한 채
크기만 조정합니다.

**두 가지 제약이 있습니다.**

1. **긴 변을 512픽셀로 맞춥니다.**
   Stable Diffusion 1.5는 512×512 이미지로 학습된 모델이라, 이 근처 크기에서 가장 안정적입니다.
   너무 크게 만들면 인물이 둘로 늘어나거나 머리가 두 개 생기는 현상이 나타납니다.

2. **가로·세로 모두 8의 배수여야 합니다.**
   Stable Diffusion은 이미지를 내부적으로 **1/8 크기로 압축한 공간(latent)** 에서 계산합니다.
   그래서 8로 나누어떨어지지 않으면 오류가 나거나 결과가 어긋납니다.

`MAX_SIDE` 값을 바꾸면 결과 해상도가 달라집니다.

| `MAX_SIDE` | 결과 | T4에서 |
| --- | --- | --- |
| `384` | 빠르지만 디테일 부족 | 매우 안전 |
| **`512`** | **기본값. 품질과 속도 균형** | **안전** |
| `640` | 얼굴 디테일이 조금 좋아짐 | 대체로 괜찮음 |
| `768` | 인물이 복제되는 현상이 자주 발생 | 메모리 부족 위험 |

In [ ]:
# [STEP 4-3] 생성할 이미지 크기 자동 계산

MAX_SIDE = 512   # 긴 변의 목표 길이. 메모리 부족 시 448 또는 384로 낮추세요.
MIN_SIDE = 384   # 짧은 변이 이보다 작아지지 않게 막습니다. (인물이 뭉개지는 것 방지)
ABS_MAX = 704    # 어떤 경우에도 넘지 않는 한 변의 상한. (메모리 폭주 · 인물 복제 방지)


def fit_size(width, height, max_side=MAX_SIDE, min_side=MIN_SIDE,
             abs_max=ABS_MAX, multiple=8):
    """원본 비율을 유지하면서 생성 크기를 정한다.
    1) 긴 변을 max_side에 맞춘다
    2) 짧은 변이 min_side보다 작아지면 min_side 기준으로 다시 키운다
    3) 그 결과 긴 변이 abs_max를 넘으면 abs_max로 되돌린다 (2보다 우선)
    4) 8의 배수로 내림한다"""
    scale = max_side / max(width, height)
    if min(width, height) * scale < min_side:
        scale = min_side / min(width, height)
    if max(width, height) * scale > abs_max:
        scale = abs_max / max(width, height)
    new_w = max(multiple, int(width * scale) // multiple * multiple)
    new_h = max(multiple, int(height * scale) // multiple * multiple)
    return new_w, new_h


WIDTH, HEIGHT = fit_size(*pose_src.size)
aspect = max(pose_src.size) / min(pose_src.size)

print(f"원본 크기   : {pose_src.size[0]} x {pose_src.size[1]}")
print(f"생성할 크기 : {WIDTH} x {HEIGHT}   <- 이 크기로 그림이 만들어집니다")
print()
print(f"8의 배수 확인 : 가로 {WIDTH} % 8 = {WIDTH % 8}, 세로 {HEIGHT} % 8 = {HEIGHT % 8}  (둘 다 0이어야 정상)")
print(f"비율 유지 확인: 원본 {pose_src.size[0] / pose_src.size[1]:.3f} -> 생성 {WIDTH / HEIGHT:.3f}")

if aspect > 1.5:
    print()
    print("!" * 66)
    print(f"경고: 가로세로비가 {aspect:.2f} 로 너무 깁니다. (16:9 화면 캡처 등)")
    print()
    print("이대로 생성하면 두 가지 문제가 생깁니다.")
    print("  1. 인물이 화면의 일부만 차지해 작게 그려집니다.")
    print("  2. 가로로 넓은 그림에서는 인물이 둘로 복제되는 현상이 자주 납니다.")
    print()
    print("해결: STEP 4-2 셀로 돌아가 인물 주변만 잘라내세요.")
    print("      (또는 STEP 5-4의 자동 크롭을 쓰세요.)")
    print("!" * 66)

---
# STEP 5 — 사진에서 자세(스켈레톤) 뽑아내기

**이 노트북의 핵심 단계입니다.**

## OpenPose가 하는 일

**OpenPose**는 사진 한 장을 보고 사람의 **관절 18곳**의 화면상 좌표를 찾아내는 모델입니다.

```
찾는 관절 18개
  코, 목
  오른쪽 어깨 / 팔꿈치 / 손목        왼쪽 어깨 / 팔꿈치 / 손목
  오른쪽 골반 / 무릎 / 발목          왼쪽 골반 / 무릎 / 발목
  오른쪽 눈 / 귀                     왼쪽 눈 / 귀
```

찾은 점들을 **부위별로 정해진 색**의 선으로 이어서, **검은 배경 위에 막대 인형 같은 그림**을 만듭니다.
이것을 **스켈레톤(skeleton)** 또는 **컨트롤 이미지**라고 부릅니다.
색깔이 부위 정보를 담고 있어서(왼팔은 초록 계열, 오른팔은 주황 계열 등)
다음 단계의 모델이 "어느 선이 어느 팔인지" 구분할 수 있습니다.

## 왜 이게 필요한가

STEP 6의 **ControlNet**에게 이 그림을 넘기면서 이렇게 지시하게 됩니다:

> "이 뼈대와 똑같은 자세로, 프롬프트에 적힌 내용을 그려라."

> ### 반드시 확인하세요
> 여기서 뼈대가 엉뚱하게 잡히면 **최종 결과도 반드시 엉망이 됩니다.**
> 출력된 스켈레톤이 원본 사진의 자세와 닮았는지 **눈으로 비교한 뒤** 넘어가세요.

## 5-0. OpenPose가 잘 못 잡는 이미지 — 먼저 읽으세요

OpenPose는 **실제 사람 사진 수십만 장으로 학습된 모델**입니다.
사람이 아닌 비율이나 사진이 아닌 화풍에서는 인식률이 뚝 떨어집니다.

| 참조 이미지 종류 | 인식률 | 대처 |
| --- | --- | --- |
| 실사 인물 사진, 전신 | 매우 좋음 | 그대로 진행 |
| 실사 인물 사진, 인물이 작음 | 보통 | **5-4 자동 크롭** |
| **애니메이션 · 일러스트** | **나쁨~보통** | **5-4 자동 크롭** 후에도 나쁘면 사진으로 교체 |
| 3D 렌더링 캐릭터 | 보통 | 5-4 자동 크롭 |
| 극단적 각도(위/아래에서) | 나쁨 | 정면·측면 이미지로 교체 |
| 팔다리가 화면 밖으로 잘림 | 잘린 부위 누락 | 전신이 보이는 이미지로 교체 |

**애니메이션 캡처가 특히 어려운 이유**

* 머리가 몸에 비해 크고 눈·머리카락이 과장되어 **실제 사람 비율과 다릅니다**
* 명암이 평평해서 팔다리의 입체감으로 관절을 추정하기 어렵습니다
* 16:9 화면 캡처는 인물이 화면의 일부만 차지하는 경우가 많습니다

애니 캡처를 쓸 거라면 **인물이 화면을 꽉 채우도록 잘라내는 것이 거의 필수**입니다.
아래 5-4 셀이 그 작업을 자동으로 해 줍니다.

## 5-1. OpenPose 모델 불러오기

인터넷에서 OpenPose 가중치 파일(약 200MB)을 내려받습니다. **첫 실행은 30초~1분** 걸립니다.

> **코드 한 줄 설명**
> `from controlnet_aux.open_pose import OpenposeDetector`
> — `controlnet_aux` 전체를 불러오면 이 노트북에서 안 쓰는 다른 분석 도구
> (깊이 추정, 윤곽선 추출 등)까지 전부 메모리에 올라가면서 오류가 날 수 있습니다.
> 필요한 `open_pose` 모듈만 콕 집어서 가져옵니다.

In [ ]:
# [STEP 5-1] OpenPose 모델 불러오기 (첫 실행 30초~1분)

from controlnet_aux.open_pose import OpenposeDetector

print("OpenPose 가중치를 내려받는 중... (첫 실행만 시간이 걸립니다)")
t0 = time.time()

# "lllyasviel/Annotators"는 ControlNet 제작자가 각종 추출 도구 가중치를 모아 둔 공식 저장소입니다.
openpose = OpenposeDetector.from_pretrained("lllyasviel/Annotators")

print(f"OpenPose 준비 완료 ({time.time() - t0:.0f}초)")

## 5-2. 관절 찾아서 스켈레톤 만들기

### 세 가지 켜고 끌 수 있는 옵션

| 옵션 | 기본값 | 켜면 | 언제 켜나 |
| --- | --- | --- | --- |
| `include_body` | `True` | 몸통·팔·다리 관절 18개 | **항상 켜 둡니다** |
| `include_hand` | `False` | 손가락 관절 21개 × 양손 | 손 모양이 중요한 포즈일 때 |
| `include_face` | `False` | 얼굴 윤곽점 70개 | 얼굴 방향·표정까지 따라 하고 싶을 때 |

**`include_hand` 를 켤지 말지**
손가락까지 잡으면 결과의 손 모양이 좋아질 수 있지만, 원본에서 손이 흐릿하거나 가려져 있으면
**엉뚱한 손가락 뼈대가 생겨 결과가 더 망가집니다.** 일단 `False`로 해 보고,
손이 뭉개지면 `True`로 바꿔 비교해 보세요.

**`detect_resolution=512`**
관절을 찾을 때 사진을 몇 픽셀로 줄여서 볼지입니다. 512면 충분하고, 올려도 크게 좋아지지 않으면서
느려집니다.

**마지막 `resize` 가 왜 필요한가**
OpenPose가 만든 스켈레톤과 STEP 4-3에서 정한 생성 크기가 **정확히 같아야** 합니다.
크기가 다르면 자세가 미묘하게 어긋난 채로 전달됩니다.

In [ ]:
# [STEP 5-2] 관절을 찾아 스켈레톤 이미지 만들기

t0 = time.time()

pose_map = openpose(
    pose_src,
    include_body=True,     # 몸통·팔·다리 (항상 True)
    include_hand=False,    # 손가락까지 잡기 (손이 뭉개지면 True로 바꿔 비교)
    include_face=False,    # 얼굴 윤곽까지 잡기 (표정을 따라 하고 싶을 때만 True)
    detect_resolution=512, # 관절을 찾을 때 볼 해상도
    output_type="pil",     # 결과를 PIL 이미지로 받기
)

# 생성 크기와 정확히 일치시킵니다.
pose_map = pose_map.resize((WIDTH, HEIGHT), Image.LANCZOS)

print(f"스켈레톤 추출 완료 ({time.time() - t0:.1f}초)")
print(f"스켈레톤 크기: {pose_map.size[0]} x {pose_map.size[1]} (생성 크기와 같아야 정상)")

## 5-3. 스켈레톤 확인 — 여기서 꼭 눈으로 보세요

관절을 하나도 못 찾으면 **완전히 검은 이미지**가 나옵니다.
그 경우를 코드가 자동으로 잡아내서 경고를 띄웁니다.

아래 `흰 픽셀 비율` 은 전체 화면에서 뼈대(색깔 선)가 차지하는 넓이입니다.
경험상 **1% 미만이면 인물이 너무 작게 잡힌 것**입니다.

In [ ]:
# [STEP 5-3] 스켈레톤 품질 확인 + 화면에 표시

arr = np.asarray(pose_map)
nonblack_ratio = (arr.max(axis=2) > 20).mean()   # 검지 않은 픽셀의 비율

print(f"뼈대가 차지하는 화면 비율: {nonblack_ratio * 100:.2f}%")
print("-" * 64)

if arr.max() == 0:
    print("실패: 관절을 하나도 찾지 못했습니다 (스켈레톤이 완전히 검은색)")
    print()
    print("원인 후보")
    print("  - 인물이 화면에서 너무 작음")
    print("  - 몸이 대부분 가려졌거나 뒷모습·극단적 각도")
    print("  - 사진에 사람이 없음")
    print()
    print("해결: STEP 4-2 셀로 돌아가 인물 주변만 잘라낸 뒤, STEP 4-3 → 5-2 → 5-3을 다시 실행하세요.")
elif nonblack_ratio < 0.01:
    print("주의: 뼈대가 너무 작게 잡혔습니다. 결과가 흐릿하거나 자세가 어긋날 수 있습니다.")
    print("      STEP 4-2로 돌아가 인물 주변만 잘라내면 크게 좋아집니다.")
else:
    print("성공: 아래 그림이 원본 사진의 자세와 닮았는지 눈으로 확인하세요.")

pose_map

### 결과 읽는 법

| 보이는 것 | 뜻 | 할 일 |
| --- | --- | --- |
| 검은 배경에 **알록달록한 막대 인형** | 성공 | STEP 6으로 |
| **완전히 검은 화면** | 인물 인식 실패 | STEP 4-2에서 인물 주변만 크롭 |
| **팔/다리 일부가 빠짐** | 원본에서 가려진 부위 | 그대로 진행 가능. 빠진 부위는 모델이 알아서 그립니다 |
| **막대가 여러 개 뒤엉킴** | 사람이 여러 명으로 인식됨 | 한 사람만 나오게 크롭 |
| **뼈대가 화면 구석에 작게** | 인물이 작음 | 크롭해서 크게 만들기 |

## 5-4. *(결과가 나쁠 때)* 인물 기준 자동 크롭 후 다시 추출

5-3의 뼈대가 **작거나 어긋났다면 이 셀을 실행하세요.** 좌표를 직접 계산할 필요가 없습니다.

### 하는 일

1. 방금 찾은 **관절 좌표로 인물을 감싸는 사각형**을 계산합니다
2. 주위에 18% 여백을 두고 **자동으로 잘라냅니다**
3. 잘린 이미지로 **스켈레톤을 다시 추출**하고, 생성 크기도 다시 계산합니다

인물이 화면을 크게 채우게 되므로 **관절 인식률과 결과 품질이 함께 올라갑니다.**
특히 16:9 화면 캡처처럼 가로로 긴 이미지에서 효과가 큽니다.

> 이 셀은 **여러 번 실행하지 마세요.** 실행할 때마다 더 좁게 잘려 들어갑니다.
> 너무 좁아졌다면 STEP 4-1부터 다시 실행해 원본을 새로 불러오세요.

In [ ]:
# [STEP 5-4] 인물 기준 자동 크롭 후 재추출 (5-3 결과가 나쁠 때만 실행)

MARGIN = 0.18   # 인물 주위에 남길 여백 비율 (0.18 = 18%). 손발이 잘리면 0.3으로 올리세요.

poses = openpose.detect_poses(pose_src)

if not poses:
    print("관절을 찾지 못해 자동 크롭을 할 수 없습니다.")
    print("-> STEP 4-2에서 직접 좌표를 정해 잘라낸 뒤 4-3 -> 5-2 -> 5-3을 다시 실행하세요.")
else:
    # 관절이 가장 많이 잡힌 사람을 고릅니다. (사람이 여럿일 때 대비)
    try:
        best = max(poses, key=lambda p: sum(1 for k in p.body.keypoints if k is not None))
        pts = [k for k in best.body.keypoints if k is not None]
        xs = [k.x for k in pts]
        ys = [k.y for k in pts]
    except AttributeError as err:
        raise RuntimeError(
            f"관절 좌표를 읽는 방식이 예상과 다릅니다 ({err}).\n"
            "이 셀 대신 STEP 4-2에서 직접 좌표를 정해 잘라내세요."
        )

    W0, H0 = pose_src.size

    # 좌표가 0~1로 정규화되어 있으면 픽셀 단위로 환산합니다.
    if max(xs) <= 1.5 and max(ys) <= 1.5:
        xs = [x * W0 for x in xs]
        ys = [y * H0 for y in ys]

    x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
    mx, my = (x1 - x0) * MARGIN, (y1 - y0) * MARGIN
    x0, x1, y0, y1 = x0 - mx, x1 + mx, y0 - my, y1 + my

    # 누운 자세처럼 상자가 극단적으로 납작하거나 길쭉하면, 짧은 축을 넓혀 비율을 완화합니다.
    # (이 보정이 없으면 누운 사람에서 1000px 넘게 가로로 긴 그림이 만들어집니다.)
    MAX_ASPECT = 1.6
    bw, bh = x1 - x0, y1 - y0
    if bw / bh > MAX_ASPECT:
        pad = (bw / MAX_ASPECT - bh) / 2
        y0, y1 = y0 - pad, y1 + pad
    elif bh / bw > MAX_ASPECT:
        pad = (bh / MAX_ASPECT - bw) / 2
        x0, x1 = x0 - pad, x1 + pad

    box = (max(0, int(x0)), max(0, int(y0)), min(W0, int(x1)), min(H0, int(y1)))

    print(f"찾은 사람 수   : {len(poses)} (관절이 가장 많은 사람 기준)")
    print(f"사용한 관절 수 : {len(pts)}개 / 18개")
    print(f"자르는 영역    : {box}")
    print(f"영역 가로세로비: {(box[2] - box[0]) / max(1, box[3] - box[1]):.2f}")
    print()

    # 자르고, 크기를 다시 계산하고, 스켈레톤을 다시 뽑습니다.
    pose_src = pose_src.crop(box)
    WIDTH, HEIGHT = fit_size(*pose_src.size)
    pose_map = openpose(pose_src, include_body=True, include_hand=False,
                        include_face=False, detect_resolution=512, output_type="pil")
    pose_map = pose_map.resize((WIDTH, HEIGHT), Image.LANCZOS)

    ratio = (np.asarray(pose_map).max(axis=2) > 20).mean()
    print(f"자른 뒤 원본   : {pose_src.size[0]} x {pose_src.size[1]}")
    print(f"생성할 크기    : {WIDTH} x {HEIGHT}")
    print(f"뼈대 화면 비율 : {ratio * 100:.2f}%  (5-3보다 커졌으면 성공)")

pose_map

### 이 뒤로는

크롭 결과가 만족스러우면 **STEP 6으로 넘어가세요.** (STEP 4-3과 5-2는 이 셀이 이미 다시 해 줬습니다.)

여전히 나쁘다면 참조 이미지 자체가 OpenPose에 어려운 경우입니다. 위 **5-0 주의사항**을 참고해
다른 사진으로 바꾸는 편이 빠릅니다.

---
# STEP 6 — 이미지 생성 모델 불러오기

이제 그림을 그릴 **모델 두 개**를 GPU에 올립니다.

| 모델 | 역할 | 크기 |
| --- | --- | --- |
| **ControlNet (OpenPose)** | "이 뼈대를 지켜라"라고 강제하는 **감독** | 약 1.4GB |
| **Stable Diffusion 1.5** | 프롬프트를 읽고 실제로 그림을 그리는 **화가** | 약 2GB |

## 두 모델이 협력하는 방식

Stable Diffusion은 원래 프롬프트만 보고 그립니다. 그래서 "팔을 머리 위로 든 자세"라고 글로 써도
**정확히 어느 각도로 들지는 통제할 수 없습니다.**

ControlNet은 Stable Diffusion 옆에 붙어서, 그림을 그리는 **매 단계마다**
"지금 그리는 것이 뼈대와 얼마나 맞는가"를 계산해 방향을 밀어 줍니다.
그래서 **글로는 표현 불가능한 정확한 자세**를 지정할 수 있게 됩니다.

> **첫 실행에는 모델 다운로드로 4~6분 걸립니다.** 진행 막대가 여러 개 지나갑니다.
> 같은 런타임에서 두 번째로 실행하면 캐시가 남아 있어 10초 안에 끝납니다.

## 6-1. 어떤 모델을 쓸지 정하기

여기서 모델 주소만 정하고, 실제 다운로드는 6-2부터 합니다.

**그림체를 바꾸고 싶다면 `BASE_MODEL` 만 바꿔서 6-1 ~ 6-3을 다시 실행**하면 됩니다.

| `BASE_MODEL` 주소 | 특징 |
| --- | --- |
| `stable-diffusion-v1-5/stable-diffusion-v1-5` | 기본. 무난한 범용 |
| `Lykon/dreamshaper-8` | 일러스트·판타지 콘셉트 아트에 강함 |
| `SG161222/Realistic_Vision_V5.1_noVAE` | 실사 사진 느낌 |

> **반드시 SD 1.5 계열 모델만** 써야 합니다. SDXL 계열 모델은 이 ControlNet과 호환되지 않습니다.

In [ ]:
# [STEP 6-1] 사용할 모델 주소 정하기

# 그림을 그리는 본체 모델.
# 주의: 예전에 널리 쓰이던 "runwayml/stable-diffusion-v1-5" 주소는 2024년에 삭제되었습니다.
#      아래가 현재 유지되고 있는 공식 미러입니다.
BASE_MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"

# 자세를 강제하는 ControlNet 모델. v1.1이 v1.0보다 정확하고 손·얼굴 입력까지 지원합니다.
CONTROLNET = "lllyasviel/control_v11p_sd15_openpose"

print(f"본체 모델   : {BASE_MODEL}")
print(f"ControlNet  : {CONTROLNET}")
print(f"자료형      : {DTYPE}")

## 6-2. 모델 내려받아 메모리에 올리기 (첫 실행 4~6분)

**코드에 나오는 설정 설명**

| 설정 | 하는 일 |
| --- | --- |
| `torch_dtype=DTYPE` | STEP 2에서 정한 `float16`으로 올려 메모리를 절반만 사용 |
| `safety_checker=None` | 선정성 검사 모델을 빼서 메모리 약 300MB와 로딩 시간 절약 |
| `requires_safety_checker=False` | 검사기를 뺐다는 사실을 명시해 불필요한 경고 억제 |

`from_pretrained` 는 "인터넷 저장소에서 미리 학습된 가중치를 내려받아 모델을 만든다"는 뜻입니다.
한 번 받은 파일은 `~/.cache/huggingface/` 에 저장되어, 같은 런타임에서는 다시 받지 않습니다.

In [ ]:
# [STEP 6-2] 모델 내려받기 (첫 실행 4~6분)

from diffusers import (
    ControlNetModel,
    StableDiffusionControlNetPipeline,
    UniPCMultistepScheduler,
)

t0 = time.time()

print("[1/2] ControlNet(자세 감독) 내려받는 중... 약 1.4GB")
controlnet = ControlNetModel.from_pretrained(CONTROLNET, torch_dtype=DTYPE)
print(f"      완료 ({time.time() - t0:.0f}초)\n")

print("[2/2] Stable Diffusion(화가) 내려받는 중... 약 2GB, 가장 오래 걸립니다")
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    BASE_MODEL,
    controlnet=controlnet,
    torch_dtype=DTYPE,
    safety_checker=None,
    requires_safety_checker=False,
)
print(f"      완료 (누적 {time.time() - t0:.0f}초)")

### 결과 읽는 법

* 진행 막대가 끝까지 가고 `완료` 가 찍히면 성공입니다.
* `You have disabled the safety checker...` 경고는 **정상**입니다. 일부러 껐기 때문에 나옵니다.
* `OSError: ... is not a local folder and is not a valid model identifier`
  → 인터넷 문제이거나 모델 주소 오타입니다. 6-1의 주소를 확인하고 6-2를 다시 실행하세요.

## 6-3. 스케줄러 교체 + GPU에 배치

### 스케줄러(scheduler)란?

Stable Diffusion은 **무작위 노이즈에서 시작해 조금씩 노이즈를 걷어내며** 그림을 만듭니다.
**"매 단계에서 노이즈를 얼마나, 어떤 식으로 걷어낼지"를 정하는 알고리즘이 스케줄러**입니다.

기본 스케줄러(PNDM)는 좋은 결과를 내려면 50단계쯤 필요하지만,
**`UniPCMultistepScheduler`** 는 **20~30단계로 같은 품질**을 냅니다.
T4처럼 느린 GPU에서 **거의 2배 빨라지는 셈**이라 바꿔 줍니다.

### 메모리 절약 옵션

| 옵션 | 효과 | 속도 |
| --- | --- | --- |
| `enable_attention_slicing()` | 계산을 잘게 쪼개 메모리 사용량 감소 | 5~10% 느려짐 |
| `enable_model_cpu_offload()` | 당장 안 쓰는 부품을 CPU로 내려 둠 | 30~50% 느려짐 |

512×512에서 T4는 메모리가 넉넉하지만, `attention_slicing` 은 보험으로 켜 둡니다.
정말 메모리 부족(OOM)이 나면 아래 `LOW_VRAM` 을 `True` 로 바꾸세요.

> 두 옵션은 **함께 쓸 수 없습니다.** `cpu_offload` 를 쓸 때는 `.to("cuda")` 를 호출하면 안 되기 때문에,
> 코드에서 `if` 로 갈라 놓았습니다.

In [ ]:
# [STEP 6-3] 스케줄러 교체 + GPU에 배치

LOW_VRAM = False   # 메모리 부족(OOM) 오류가 계속 나면 True로 바꾸고 이 셀만 다시 실행하세요.

# 스케줄러 교체: 적은 단계로 같은 품질을 내는 UniPC로 바꿉니다.
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

if LOW_VRAM:
    # 안 쓰는 부품을 CPU에 내려 두었다가 필요할 때만 GPU로 올립니다. 느리지만 메모리를 아낍니다.
    pipe.enable_model_cpu_offload()
    print("저메모리 모드로 배치했습니다. (느리지만 안전)")
else:
    pipe = pipe.to(DEVICE)              # 모델 전체를 GPU로
    pipe.enable_attention_slicing()     # 계산을 쪼개 메모리 여유 확보
    print(f"{DEVICE.upper()}에 배치 완료.")

print(f"스케줄러: {type(pipe.scheduler).__name__}")

if DEVICE == "cuda":
    used = torch.cuda.memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU 메모리 사용량: {used:.2f} GB / {total:.1f} GB")

print("\n모델 준비 완료 — STEP 7로 넘어가세요.")

### 결과 읽는 법

* `모델 준비 완료` 와 메모리 사용량(보통 **2.5~3.5GB**)이 보이면 성공입니다.
* `LOW_VRAM = True` 로 바꿨다면 사용량이 **0GB 근처**로 나옵니다. 정상입니다.
  (필요할 때만 GPU에 올리는 방식이라 대기 중에는 비어 있습니다.)

---
# STEP 7 — 무엇을 그릴지 정하기

**이 노트북에서 여러분이 실제로 계속 손대게 될 부분입니다.**
STEP 7 → STEP 8 → STEP 9 를 반복하며 원하는 결과를 찾아 가면 됩니다.

## 7-1. 프롬프트 — 무엇을 그릴까

**자세는 이미 스켈레톤이 책임지고 있습니다.** 그러니 프롬프트에는 **나머지 전부**를 씁니다.

```
[누가] + [무엇을 입고] + [어디서] + [어떤 화풍 / 조명]
```

예시를 분해하면:

| 조각 | 역할 |
| --- | --- |
| `a knight in ornate silver armor` | 누가 · 무엇을 입고 |
| `one arm raised overhead` | 강조하고 싶은 동작 (선택) |
| `windswept red cape` | 세부 소품 |
| `misty battlefield at dawn` | 어디서 |
| `fantasy concept art, dramatic rim light` | 화풍 · 조명 |
| `highly detailed` | 품질 강조 단어 |

### 규칙

* **반드시 영어로 씁니다.** 이 모델은 영어 데이터로 학습되어 한국어는 거의 알아듣지 못합니다.
* **쉼표로 짧은 표현을 나열**하는 방식이 긴 문장보다 훨씬 잘 먹힙니다.
* **앞에 쓴 단어일수록 강하게 반영**됩니다. 중요한 것을 앞에 두세요.
* 참조 자세에서 특히 살리고 싶은 동작은 프롬프트에 **한 번 더** 적어 주면 안정적입니다.
* 사람이 여러 명 나오면 맨 앞에 `solo,` 를 붙이세요.
* 프롬프트는 약 **77개 토큰(대략 단어 60개)** 까지만 반영됩니다. 그 뒤는 잘립니다.

In [ ]:
# [STEP 7-1] 프롬프트 — 무엇을 그릴지 (영어로!)

PROMPT = (
    "solo, a knight in ornate silver armor, one arm raised overhead, "
    "windswept red cape, misty battlefield at dawn, "
    "fantasy concept art, dramatic rim light, highly detailed"
)

print("프롬프트:")
print(f"  {PROMPT}")
print(f"\n대략 단어 수: {len(PROMPT.split())}개 (60개 넘으면 뒷부분이 무시될 수 있습니다)")

## 7-2. 네거티브 프롬프트 — 무엇을 피할까

**"이런 건 그리지 마라"** 목록입니다. 모델을 반대 방향으로 밀어내는 역할을 합니다.
손가락이 뭉개지거나 팔이 하나 더 생기는 흔한 실패를 눈에 띄게 줄여 줍니다.

| 넣은 단어 | 막으려는 실패 |
| --- | --- |
| `bad anatomy, extra limbs, missing limbs` | 팔다리가 더 생기거나 사라짐 |
| `extra fingers, fused fingers, deformed hands` | 손가락 개수 이상, 손 뭉개짐 |
| `deformed face, mutated` | 얼굴 붕괴 |
| `watermark, text, signature` | 워터마크·글자가 그려짐 |
| `lowres, blurry, jpeg artifacts` | 흐릿함, 압축 노이즈 |

아래 기본값을 그대로 써도 충분합니다. 특정 요소를 빼고 싶으면 뒤에 단어를 덧붙이세요.
(예: 모자를 원치 않으면 `, hat`)

In [ ]:
# [STEP 7-2] 네거티브 프롬프트 — 무엇을 피할지

NEGATIVE_PROMPT = (
    "lowres, bad anatomy, extra limbs, missing limbs, extra fingers, "
    "fused fingers, deformed hands, deformed face, mutated, "
    "watermark, text, signature, jpeg artifacts, blurry"
)

print("네거티브 프롬프트:")
print(f"  {NEGATIVE_PROMPT}")

## 7-3. 숫자 설정

| 변수 | 뜻 | 추천 | 올리면 | 내리면 |
| --- | --- | --- | --- | --- |
| `STEPS` | 노이즈를 걷어내는 횟수 | **25~30** | 정교해짐 · 느려짐 | 거칠어짐 · 빨라짐 |
| `GUIDANCE` | 프롬프트를 얼마나 곧이곧대로 따를지 | **7~9** | 프롬프트 충실 · 색이 타 버림 | 자유롭고 자연스러움 · 프롬프트 무시 |
| `POSE_STRENGTH` | **참조 자세를 얼마나 엄격히 지킬지** | **1.0** | 자세 정확 · 그림이 뻣뻣해짐 | 자연스러움 · 자세가 흐트러짐 |
| `SEED` | 무작위 시작점 번호 | 아무 정수 | — | — |

### `SEED` 를 제대로 쓰는 법

같은 `SEED` + 같은 설정 = **몇 번을 돌려도 완전히 똑같은 그림**이 나옵니다.

* **설정을 비교할 때**: `SEED` 를 고정하고 `GUIDANCE` 나 `POSE_STRENGTH` 만 바꾸면
  **그 설정의 효과만** 순수하게 볼 수 있습니다.
* **마음에 드는 걸 고를 때**: 설정을 고정하고 `SEED` 만 바꿔 여러 장 뽑습니다.
  특히 **손가락과 얼굴은 운이 크게 작용**해서, 이 방법이 가장 빠릅니다. (STEP 10 참고)
* **좋은 결과가 나오면 `SEED` 값을 꼭 적어 두세요.** 그래야 나중에 재현할 수 있습니다.

### `STEPS` 와 소요 시간 (T4 / 512×512 기준)

| `STEPS` | 대략 소요 시간 |
| --- | --- |
| 20 | 약 8초 |
| 30 | 약 12초 |
| 50 | 약 20초 |

In [ ]:
# [STEP 7-3] 숫자 설정

STEPS = 30            # 생성 단계 수 (20~30 권장)
GUIDANCE = 7.5        # 프롬프트 충실도 (7~9 권장)
POSE_STRENGTH = 1.0   # 자세 엄격도 (0.8 느슨함 ~ 1.2 엄격함)
SEED = 12345          # 같은 숫자 = 항상 같은 그림

print(f"STEPS         = {STEPS}")
print(f"GUIDANCE      = {GUIDANCE}")
print(f"POSE_STRENGTH = {POSE_STRENGTH}")
print(f"SEED          = {SEED}")

## 7-4. 저장 위치 정하기 + 최종 설정 확인

결과는 `outputs/` 폴더에 저장됩니다. 왼쪽 사이드바의 **폴더 아이콘 📁** 을 누르면 직접 볼 수 있습니다.

> **중요**: Colab에 저장한 파일은 **런타임이 끊기면 모두 사라집니다.**
> 마음에 드는 결과는 STEP 9-3에서 **반드시 내 컴퓨터로 내려받으세요.**

In [ ]:
# [STEP 7-4] 저장 위치 + 전체 설정 요약

# 프로젝트 폴더를 찾았다면 그 안에 저장합니다.
# -> 구글 드라이브에 저장되므로 내 PC의 프로젝트 폴더로 자동 동기화되고,
#    런타임이 끊겨도 결과가 사라지지 않습니다.
if "PROJECT_DIR" in globals() and os.path.isdir(PROJECT_DIR):
    OUT_DIR = os.path.join(PROJECT_DIR, "outputs")
    SAVE_NOTE = "구글 드라이브 (내 PC로 자동 동기화됨)"
else:
    OUT_DIR = "outputs"
    SAVE_NOTE = "임시 서버 (런타임이 끊기면 사라짐 -> STEP 9-3에서 꼭 내려받으세요)"

os.makedirs(OUT_DIR, exist_ok=True)

print("=" * 66)
print("생성 설정 최종 확인")
print("=" * 66)
print(f"프롬프트      : {PROMPT}")
print(f"네거티브      : {NEGATIVE_PROMPT[:56]}...")
print(f"이미지 크기   : {WIDTH} x {HEIGHT}")
print(f"단계 수       : {STEPS}")
print(f"프롬프트 충실도: {GUIDANCE}")
print(f"자세 엄격도   : {POSE_STRENGTH}")
print(f"seed          : {SEED}")
print(f"저장 폴더     : {OUT_DIR}")
print(f"              ({SAVE_NOTE})")
print("=" * 66)
print("STEP 8에서 생성합니다.")

---
# STEP 8 — 이미지 생성

## 내부에서 무슨 일이 일어나나요?

1. **완전한 무작위 노이즈**(모래알 같은 화면)에서 시작합니다.
2. 매 단계마다 두 가지를 동시에 고려해 노이즈를 조금씩 걷어냅니다.
   * 프롬프트에 가까워지도록 (Stable Diffusion)
   * 스켈레톤 자세에 맞도록 (ControlNet)
3. `STEPS` 번 반복하면 노이즈가 사라지고 그림이 남습니다.
4. 마지막으로 **VAE**라는 부품이 압축 공간의 결과를 실제 픽셀 이미지로 펼칩니다.

**T4에서 512×512 / 30단계 기준 10~20초** 걸립니다.
진행 막대가 `0/30` 부터 `30/30` 까지 올라갑니다.

## 8-1. 생성 실행

`torch.Generator(...).manual_seed(SEED)` 는 난수의 출발점을 고정하는 장치입니다.
이것 때문에 같은 `SEED` 로는 항상 같은 그림이 나옵니다.

In [ ]:
# [STEP 8-1] 이미지 생성 (10~20초)

generator = torch.Generator(device=DEVICE).manual_seed(SEED)

t0 = time.time()

result = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    image=pose_map,                                # <- STEP 5에서 만든 스켈레톤
    width=WIDTH,
    height=HEIGHT,
    num_inference_steps=STEPS,
    guidance_scale=GUIDANCE,
    controlnet_conditioning_scale=POSE_STRENGTH,   # <- 자세를 얼마나 지킬지
    generator=generator,
).images[0]

elapsed = time.time() - t0
print(f"\n생성 완료 — {elapsed:.1f}초 (단계당 {elapsed / STEPS:.2f}초)")

## 8-2. 결과 보기

In [ ]:
# [STEP 8-2] 생성된 이미지 보기

print(f"크기: {result.size[0]} x {result.size[1]}  |  seed: {SEED}  |  자세 엄격도: {POSE_STRENGTH}")
result

### 결과가 마음에 안 들 때 — 무엇부터 바꿀까

**순서대로 시도하세요.** 바꾼 뒤에는 **해당 STEP 7 셀 → STEP 8-1 → 8-2** 만 다시 실행하면 됩니다.
모델은 이미 메모리에 있으니 STEP 6을 다시 할 필요가 없습니다.

| 증상 | 바꿀 것 | 어떻게 |
| --- | --- | --- |
| **자세가 참조와 다르다** | `POSE_STRENGTH` (7-3) | `1.0` → `1.2` 로 올리기 |
| **자세는 맞는데 뻣뻣하다** | `POSE_STRENGTH` (7-3) | `1.0` → `0.8` 로 내리기 |
| **손·얼굴만 이상하다** | `SEED` (7-3) | 아무 다른 숫자로. 3~4번 반복 |
| **프롬프트 내용이 무시된다** | `GUIDANCE` (7-3) | `7.5` → `9` 로 올리고, 중요한 단어를 프롬프트 앞으로 |
| **색이 타 버린 듯 과장된다** | `GUIDANCE` (7-3) | `7.5` → `6` 으로 내리기 |
| **전체적으로 거칠다** | `STEPS` (7-3) | `30` → `40` 으로 올리기 |
| **사람이 여러 명 나온다** | `PROMPT` (7-1) | 맨 앞에 `solo,` 추가 / `MAX_SIDE` 를 512로 낮추기 |
| **원치 않는 물건이 나온다** | `NEGATIVE_PROMPT` (7-2) | 그 단어를 네거티브에 추가 |

### 오류가 났을 때

* `torch.cuda.OutOfMemoryError` — GPU 메모리 부족. 아래 순서로 해결하세요.
  1. 상단 메뉴 **런타임 → 세션 다시 시작** 후 STEP 1부터 다시 실행
  2. 그래도 나면 STEP 4-3의 `MAX_SIDE` 를 `384` 로 낮추기
  3. STEP 6-3의 `LOW_VRAM` 을 `True` 로 바꾸고 6-3만 다시 실행
* `NameError: name 'pipe' is not defined` — 런타임이 끊긴 것입니다. STEP 1부터 다시 실행하세요.

---
# STEP 9 — 저장 · 비교 · 다운로드

## 9-1. 파일로 저장

In [ ]:
# [STEP 9-1] 결과를 파일로 저장

# 파일 이름에 설정값을 넣습니다.  예: result_seed12345_cn1.0_g7.5.png
# 같은 이름이 이미 있으면 뒤에 -1, -2 ... 를 붙여 기존 파일을 덮어쓰지 않습니다.
BASE_NAME = f"result_seed{SEED}_cn{POSE_STRENGTH}_g{GUIDANCE}"

OUT_PATH = os.path.join(OUT_DIR, f"{BASE_NAME}.png")
dup = 0
while os.path.exists(OUT_PATH):
    dup += 1
    OUT_PATH = os.path.join(OUT_DIR, f"{BASE_NAME}-{dup}.png")

result.save(OUT_PATH)

print(f"저장 완료: {OUT_PATH}")
print(f"           ({os.path.getsize(OUT_PATH) / 1024:.0f} KB)")
if dup:
    print(f"           같은 설정의 파일이 이미 있어 -{dup} 을 붙였습니다.")
print()

files_now = sorted(os.listdir(OUT_DIR))
print(f"{OUT_DIR} 에 쌓인 파일 {len(files_now)}개 (최근 8개):")
for name in files_now[-8:]:
    print(f"  {name}")

## 9-2. 참조 · 스켈레톤 · 결과 나란히 비교

세 장을 나란히 놓고 보면 **자세가 제대로 옮겨졌는지 한눈에** 판단할 수 있습니다.

* 1번(참조)과 2번(스켈레톤)이 다르다 → **STEP 5 문제**. 사진을 크롭해서 다시 추출하세요.
* 2번(스켈레톤)과 3번(결과)이 다르다 → **STEP 7 문제**. `POSE_STRENGTH` 를 올리세요.

아래 코드는 STEP 1-2에서 설치한 한글 폰트를 찾아 그래프에 등록합니다.
폰트를 못 찾으면 자동으로 영어 제목으로 바뀝니다.

In [ ]:
# [STEP 9-2] 세 장 나란히 비교

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

# STEP 1-2에서 설치한 나눔고딕을 찾아 등록합니다. (없으면 영어 제목으로 자동 전환)
KOREAN_FONT = None
for path in fm.findSystemFonts():
    if "Nanum" in path and path.endswith((".ttf", ".otf")):
        fm.fontManager.addfont(path)
        KOREAN_FONT = fm.FontProperties(fname=path).get_name()
        break

if KOREAN_FONT:
    plt.rcParams["font.family"] = KOREAN_FONT
    plt.rcParams["axes.unicode_minus"] = False
    titles = ["1. 참조 사진\n(자세를 가져올 원본)",
              "2. 스켈레톤\n(추출한 관절 구조)",
              "3. 생성 결과\n(프롬프트대로 그린 그림)"]
else:
    titles = ["1. Reference\n(source photo)",
              "2. Skeleton\n(extracted pose)",
              "3. Result\n(generated)"]
    print("한글 폰트를 찾지 못해 영어 제목으로 표시합니다. (STEP 1-2 참고)")

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
for ax, img, title in zip(axes, [pose_src, pose_map, result], titles):
    ax.imshow(img)
    ax.set_title(title, fontsize=12)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 9-3. *(선택)* 내 컴퓨터로 내려받기

**3-1로 드라이브를 연결했다면 이 셀은 실행하지 않아도 됩니다.**
결과가 이미 프로젝트 폴더의 `outputs/` 안에 저장되어 있고, 구글 드라이브가 **내 PC로 자동 동기화**합니다.
잠시 뒤 탐색기에서 프로젝트 폴더의 `outputs` 를 열어 보세요.

드라이브를 연결하지 않고 3-2(URL)로 진행했다면 결과가 **임시 서버에만** 있습니다.
런타임이 끊기면 사라지므로 이 셀로 내려받으세요.

In [ ]:
# [STEP 9-3] 결과 이미지 내려받기

if IN_COLAB:
    from google.colab import files

    print("브라우저 다운로드가 시작됩니다. (팝업이 차단되면 허용해 주세요)")
    files.download(OUT_PATH)
else:
    print(f"로컬 환경입니다. 파일 위치: {os.path.abspath(OUT_PATH)}")

---
## 9-4. 설정을 바꿔 가며 실험하는 법 — VS Code 사용법

한 번 전체 실행을 끝냈다면, 이제 **STEP 7-3 → 8-1 → 9-1 세 칸만** 반복하면 됩니다.
모델은 메모리에 남아 있으므로 앞 단계를 다시 할 필요가 없습니다.

### 셀 실행 단축키

| 단축키 | 동작 | 언제 |
| --- | --- | --- |
| **`Ctrl` + `Enter`** | **이 셀만 실행하고 제자리에 머무름** | 실험 반복에 이걸 쓰세요 |
| `Shift` + `Enter` | 실행하고 아래 셀로 이동 | 처음 쭉 훑을 때 |
| — | 셀 왼쪽 **▶** 버튼 클릭 | 단축키 대신 |

셀 안의 코드를 고치려면 **회색 코드 영역을 클릭**하면 커서가 생깁니다. 고친 뒤 `Ctrl`+`Enter`.

### 반복 순서

```
  1. STEP 7-3 셀에서 숫자를 고친다  (예: SEED = 12345 -> 777)
  2. STEP 7-3 에서 Ctrl+Enter      <- 바꾼 값을 반영
  3. STEP 8-1 에서 Ctrl+Enter      <- 생성 (10~20초)
  4. STEP 8-2 에서 Ctrl+Enter      <- 결과 보기
  5. STEP 9-1 에서 Ctrl+Enter      <- 저장 (파일명에 설정값이 들어감)
```

**다시 실행하지 않아도 되는 것**: STEP 0~6 (설치·모델 로딩), STEP 7-4 (저장 폴더 설정).

> **주의**: 7-3의 값을 고치기만 하고 **7-3 셀을 실행하지 않으면** 바뀐 값이 반영되지 않습니다.
> 파이썬은 셀을 실행하는 순간에만 변수를 갱신합니다. 고쳤으면 반드시 그 셀부터 실행하세요.

### 무엇부터 실험할까

| 실험 | 고칠 값 (STEP 7-3) | 방법 |
| --- | --- | --- |
| **마음에 드는 그림 찾기** | `SEED` | 아무 숫자로 4~5번 바꿔 보고 고르기 |
| **자세 강도 조절** | `POSE_STRENGTH` | `0.8` / `1.0` / `1.2` 세 번 돌려 비교 |
| **프롬프트 충실도** | `GUIDANCE` (7-3) | `6` / `7.5` / `9` 비교 |
| **다른 장면** | `PROMPT` (7-1) | 7-1을 고치면 **7-1도 실행**해야 반영됩니다 |

`SEED`를 고정하고 다른 값 하나만 바꾸면 **그 값의 효과만** 순수하게 볼 수 있습니다.

저장 파일 이름에 설정이 들어가므로(`result_seed777_cn1.2_g7.5.png`) 나중에 어떤 설정이었는지
파일명만 봐도 알 수 있습니다.

### 여러 장을 한 번에 돌리고 싶다면

위 과정을 손으로 반복하는 대신 **STEP 10**을 쓰세요. 목록에 적어 둔 만큼 자동으로 연속 생성합니다.

---
# STEP 10 — 같은 자세로 여러 장 한 번에 뽑기 *(선택)*

**같은 스켈레톤**을 쓰면서 프롬프트와 seed만 바꿔 여러 장을 연속으로 만듭니다.
모델은 이미 메모리에 있으므로 **한 장당 10~20초씩만** 더 걸립니다.

## 이럴 때 씁니다

1. **같은 포즈로 캐릭터만 바꿔 보기** — 기사 / 댄서 / 우주비행사 …
2. **프롬프트는 고정하고 seed만 바꿔** 손·얼굴이 잘 나온 것 고르기
3. **설정 비교** — 프롬프트와 seed를 고정하고 `POSE_STRENGTH` 만 바꿔 효과 확인

## 10-1. 만들 목록 정하기

`VARIANTS` 에 `("프롬프트", seed)` 형태로 원하는 만큼 줄을 추가하세요.

In [ ]:
# [STEP 10-1] 만들 목록 정하기

# ("프롬프트", seed) 형태로 원하는 만큼 줄을 추가하세요.
VARIANTS = [
    ("solo, a knight in ornate silver armor, misty battlefield, fantasy concept art", 12345),
    ("solo, a street dancer in an oversized hoodie, neon city street at night, cinematic photo", 999),
    ("solo, an astronaut in a white spacesuit, surface of Mars, red dust, sci-fi illustration", 2024),
]

# 참고: 같은 프롬프트로 seed만 바꿔 고르고 싶다면 아래처럼 쓰면 됩니다.
# VARIANTS = [(PROMPT, s) for s in [1, 2, 3, 4]]

print(f"총 {len(VARIANTS)}장을 만듭니다. 예상 소요 시간: 약 {len(VARIANTS) * 15}초\n")
for i, (p, s) in enumerate(VARIANTS, start=1):
    print(f"  {i}. seed={s:<8} {p[:60]}...")

## 10-2. 연속 생성

In [ ]:
# [STEP 10-2] 연속 생성

results = []
t_all = time.time()

for i, (prompt, seed) in enumerate(VARIANTS, start=1):
    print(f"[{i}/{len(VARIANTS)}] seed={seed} | {prompt[:55]}...")

    g = torch.Generator(device=DEVICE).manual_seed(seed)
    img = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE_PROMPT,
        image=pose_map,
        width=WIDTH,
        height=HEIGHT,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        controlnet_conditioning_scale=POSE_STRENGTH,
        generator=g,
    ).images[0]

    path = os.path.join(OUT_DIR, f"variant_{i:02d}.png")
    img.save(path)
    results.append((img, f"#{i}  seed={seed}"))

print(f"\n{len(results)}장 저장 완료 ({time.time() - t_all:.0f}초) -> {OUT_DIR}/ 폴더")

# 만든 그림을 한 줄로 늘어놓고 봅니다.
fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 6))
if len(results) == 1:
    axes = [axes]
for ax, (img, title) in zip(axes, results):
    ax.imshow(img)
    ax.set_title(title, fontsize=12)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 10-3. 결과 전부 zip으로 묶어 내려받기

In [ ]:
# [STEP 10-3] outputs 폴더를 zip으로 묶어 내려받기

zip_base = "pose_tool_outputs"
zip_path = shutil.make_archive(zip_base, "zip", OUT_DIR)

n_files = len(os.listdir(OUT_DIR))
print(f"{n_files}개 파일을 {zip_path} 로 묶었습니다. ({os.path.getsize(zip_path) / 1024:.0f} KB)")

if IN_COLAB:
    from google.colab import files

    files.download(zip_path)
    print("다운로드가 시작됩니다.")
else:
    print(f"로컬 환경입니다. 위치: {os.path.abspath(zip_path)}")

---
# 부록 A — 문제 해결

## A-1. 오류 메시지가 뜰 때

| 오류 메시지 | 원인 | 해결 방법 |
| --- | --- | --- |
| `nvidia-smi: command not found` | GPU가 꺼져 있음 | STEP 0-1 — 런타임 유형을 T4 GPU로 |
| `ERROR: pip's dependency resolver ... gradio ... huggingface-hub ... incompatible` | **정상 동작**. 안 쓰는 gradio와의 버전 충돌 경고 | 무시. 1-3 셀 결과만 확인 |
| `ModuleNotFoundError: No module named 'diffusers'` | 설치 셀을 안 돌림 | STEP 1-1 실행 |
| `ModuleNotFoundError: No module named 'controlnet_aux'` | 설치 셀을 안 돌림 | STEP 1-1 실행 |
| `RuntimeError: POSE_IMAGE가 없습니다` | STEP 3을 건너뜀 | STEP 3-1 또는 3-2 실행 |
| `FileNotFoundError` | 경로 오타 또는 드라이브 미연결 | STEP 3-1 다시 실행 |
| **3-3 셀이 끝나지 않고 계속 `[*]`** | VS Code에서 `files.upload()` 는 동작하지 않음 | 정지 버튼으로 중단 후 **3-1** 또는 **3-2** 사용 |
| **`drive.mount()` 에서 멈춤** | VS Code에서 구글 인증창이 뜨지 않음 | 정지 버튼으로 중단 후 **3-2 (URL)** 사용 |
| `MessageError: credential propagation was unsuccessful` | 드라이브 인증 실패 | **3-2 (URL)** 사용 |
| `NameError: name 'pose_map' is not defined` | STEP 5를 건너뜀 | STEP 5-1 → 5-2 → 5-3 실행 |
| `NameError: name 'pipe' is not defined` | 런타임이 끊겼다 다시 붙음 | 런타임 → 세션 다시 시작 및 모두 실행 |
| `torch.cuda.OutOfMemoryError` | GPU 메모리 부족 | ① 세션 다시 시작 ② `MAX_SIDE=384` ③ `LOW_VRAM=True` |
| `OSError: ... is not a local folder` | 모델 다운로드 실패 | 인터넷 확인 후 STEP 6-2 다시 실행 |
| `ValueError: height and width must be divisible by 8` | 크기 계산을 건너뜀 | STEP 4-3 실행 |
| 그래프 제목이 `□□□` | 한글 폰트 없음 | STEP 1-2 실행 후 9-2 다시 실행 |

## A-2. 오류는 없는데 결과가 이상할 때

| 증상 | 원인 | 해결 방법 |
| --- | --- | --- |
| 자세가 참조와 전혀 다르다 | 자세 강도가 약함 | `POSE_STRENGTH` → `1.1`~`1.3` |
| 자세는 맞는데 뻣뻣하다 | 자세 강도가 과함 | `POSE_STRENGTH` → `0.7`~`0.9` |
| 스켈레톤이 완전히 검은색 | 인물 인식 실패 | STEP 4-2에서 인물 주변만 크롭 |
| 손가락이 뭉개진다 | OpenPose가 손가락을 안 잡음 | `SEED` 바꿔 여러 장(STEP 10) / `include_hand=True` 시도 |
| 얼굴이 무너진다 | 얼굴 영역이 너무 작음 | 상반신 위주 참조 사진 / `MAX_SIDE=640` |
| 팔다리가 화면 밖으로 잘린다 | 비율 불일치 | STEP 4-3이 자동 처리. 원본을 인물에 맞게 크롭하면 더 좋음 |
| 프롬프트 내용이 무시된다 | 충실도가 낮음 | `GUIDANCE` → `9`~`11`, 중요 단어를 앞으로 |
| 색이 타 버린 듯 과장된다 | 충실도가 과함 | `GUIDANCE` → `6`~`7` |
| 사람이 여러 명 나온다 | 해상도가 큼 / 프롬프트 | `MAX_SIDE=512`, 프롬프트 앞에 `solo,` |
| 머리가 두 개 생긴다 | 해상도가 너무 큼 | `MAX_SIDE` 를 `512` 로 낮추기 |

## A-3. 자주 묻는 질문

**Q. 런타임이 끊겼습니다. 처음부터 다 해야 하나요?**
네. Colab은 일정 시간 조작이 없으면 연결을 끊고 설치 내용과 파일을 모두 지웁니다.
**런타임 → 세션 다시 시작 및 모두 실행** 을 누르면 STEP 1부터 자동으로 돌아갑니다.
(참조 사진은 STEP 3-1에서 드라이브를 다시 연결해야 합니다.)

**Q. 무료 GPU 사용 시간에 제한이 있나요?**
있습니다. 하루 몇 시간 정도 쓰면 "사용량 한도 초과" 안내가 나오고 몇 시간 뒤 풀립니다.
이 노트북 정도의 작업이면 보통 문제가 되지 않습니다.

**Q. 한국어 프롬프트를 쓰면 안 되나요?**
쓸 수는 있지만 거의 무시됩니다. 이 모델은 영어 데이터로 학습되었습니다. 영어로 쓰세요.

**Q. 참조 사진 속 사람의 얼굴이 결과에 나오나요?**
아니요. 관절 좌표만 추출하므로 **얼굴·피부색·옷 등 외모 정보는 전혀 전달되지 않습니다.**

**Q. 더 큰 이미지를 만들고 싶습니다.**
`MAX_SIDE` 를 `640` 까지는 올려 볼 만합니다. 그 이상에서는 인물이 복제되는 현상이 자주 생기므로,
512로 만든 뒤 별도의 업스케일 도구로 키우는 편이 안전합니다.

**Q. 같은 결과를 나중에 다시 만들려면 무엇을 적어 둬야 하나요?**
`BASE_MODEL`, `PROMPT`, `NEGATIVE_PROMPT`, `WIDTH`/`HEIGHT`, `STEPS`, `GUIDANCE`,
`POSE_STRENGTH`, `SEED`, 그리고 **참조 사진 파일** 입니다. 이 전부가 같으면 똑같은 그림이 나옵니다.

---

# 부록 B — 용어 사전

| 용어 | 뜻 |
| --- | --- |
| **Stable Diffusion** | 텍스트를 받아 이미지를 만드는 오픈소스 모델. 여기서는 1.5 버전을 씁니다. |
| **ControlNet** | Stable Diffusion에 "구조 조건"을 추가로 주는 보조 모델. 여기서는 자세를 강제합니다. |
| **OpenPose** | 사진에서 사람의 관절 위치를 찾아내는 모델. |
| **스켈레톤 / 컨트롤 이미지** | 관절을 점과 선으로 그린 검은 배경 이미지. ControlNet의 입력. |
| **프롬프트** | 무엇을 그릴지 적는 글. 영어로 씁니다. |
| **네거티브 프롬프트** | 그리지 말아야 할 것을 적는 글. |
| **seed** | 무작위 생성의 출발점 번호. 같으면 같은 결과가 나옵니다. |
| **steps** | 노이즈를 걷어내는 반복 횟수. |
| **guidance scale (CFG)** | 프롬프트를 얼마나 강하게 따를지의 배율. |
| **scheduler** | 매 단계 노이즈를 얼마나 걷어낼지 정하는 알고리즘. |
| **latent (잠재 공간)** | 이미지를 1/8 크기로 압축한 내부 표현. 계산은 여기서 이루어집니다. |
| **VAE** | latent를 실제 픽셀 이미지로 펼쳐 주는 부품. |
| **파이프라인(pipeline)** | 위 부품들을 묶어 "프롬프트 → 이미지" 한 번에 처리하게 만든 묶음. |
| **fp16 / float16** | 숫자를 16비트로 저장하는 방식. 메모리 절반, 속도 2배. |
| **OOM** | Out Of Memory. GPU 메모리 부족 오류. |
| **CUDA** | NVIDIA GPU에서 계산을 돌리기 위한 규격의 이름. |

---

# 부록 C — 다음에 해 볼 것

* **실험 기록 남기기** — 좋은 결과가 나오면 프롬프트와 설정을 [`prompts.md`](prompts.md) 에 적어 두세요.
  나중에 재현하려면 seed까지 필요합니다.
* **그림체 바꾸기** — STEP 6-1의 `BASE_MODEL` 을 바꿔 보세요.
* **표정까지 따라 하기** — STEP 5-2에서 `include_face=True` 로 바꾸면
  참조 사진의 얼굴 방향과 표정까지 반영됩니다.
* **손 모양까지 따라 하기** — STEP 5-2에서 `include_hand=True`.
  단, 원본에서 손이 선명하게 보일 때만 효과가 있습니다.
* **다른 ControlNet 써 보기** — 자세 대신 **윤곽선**(`control_v11p_sd15_canny`)이나
  **깊이**(`control_v11f1p_sd15_depth`)를 조건으로 주면 전혀 다른 통제가 가능합니다.